# Credit Card Fraud Detection — Meta-Learner & Evaluation

Loads base model predictions from `base_models.ipynb` and trains six meta-learners on the stacked OOF features.

**Run `base_models.ipynb` first** to generate the `.npy` files this notebook depends on.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, IsolationForest,
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve,
    confusion_matrix,
)
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Load Base Model Outputs

In [ ]:
# Labels
y_train = np.load("y_train.npy")
y_val   = np.load("y_val.npy")
y_test  = np.load("y_test.npy")

# FFN
ffn_train_oof  = np.load("ffn_train_oof.npy")
ffn_val_probs  = np.load("ffn_val_probs.npy")
ffn_test_probs = np.load("ffn_test_probs.npy")

# XGBoost
xgb_train_oof  = np.load("xgb_train_oof.npy")
xgb_val_probs  = np.load("xgb_val_probs.npy")
xgb_test_probs = np.load("xgb_test_probs.npy")

# Autoencoder
ae_train_probs = np.load("ae_train_probs.npy")
ae_val_probs   = np.load("ae_val_probs.npy")
ae_test_probs  = np.load("ae_test_probs.npy")

# Isolation Forest
if_train_probs = np.load("if_train_probs.npy")
if_val_probs   = np.load("if_val_probs.npy")
if_test_probs  = np.load("if_test_probs.npy")

print(f"Train: {len(y_train):,}  ({int(y_train.sum())} fraud)")
print(f"Val:   {len(y_val):,}   ({int(y_val.sum())} fraud)")
print(f"Test:  {len(y_test):,}   ({int(y_test.sum())} fraud)")
print("\nAll base model outputs loaded.")

## 2. Meta-Learner Comparison

The meta-learner sees **4 features** per sample: `[FFN_prob, XGB_prob, AE_anomaly, IF_anomaly]`.

| Split | FFN / XGB source | AE / IF source | Used for |
|---|---|---|---|
| **meta_X_train** | OOF predictions (no leakage) | Direct predictions on X_train | Train meta-learner weights |
| **meta_X_val** | Full-retrain predictions | Direct predictions on X_val | Tune threshold |
| **meta_X_test** | Full-retrain predictions | Direct predictions on X_test | Final evaluation |

In [ ]:
meta_X_train = np.column_stack([ffn_train_oof,  xgb_train_oof,  ae_train_probs, if_train_probs])
meta_X_val   = np.column_stack([ffn_val_probs,  xgb_val_probs,  ae_val_probs,   if_val_probs])
meta_X_test  = np.column_stack([ffn_test_probs, xgb_test_probs, ae_test_probs,  if_test_probs])

print(f"meta_X_train: {meta_X_train.shape}  ({int(y_train.sum())} fraud)")
print(f"meta_X_val:   {meta_X_val.shape}")
print(f"meta_X_test:  {meta_X_test.shape}")
print(f"Features: [FFN_prob, XGB_prob, AE_anomaly, IF_anomaly]\n")

meta_learner_defs = {
    "LR":       LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED),
    "RF":       RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1),
    "GBM":      GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    "SVM":      SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED),
    "MLP":      MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=SEED),
}

meta_results = {}
for name, clf in meta_learner_defs.items():
    clf.fit(meta_X_train, y_train)
    meta_results[name] = {
        "val_probs":  clf.predict_proba(meta_X_val)[:, 1],
        "test_probs": clf.predict_proba(meta_X_test)[:, 1],
    }
    print(f"  Trained: Meta-{name}")

print("\nAll meta-learners ready.")

## 3. Evaluation

In [ ]:
def best_threshold_f1(probs, labels):
    thresholds = np.linspace(0.01, 0.99, 200)
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


def evaluate(name, test_probs, val_probs, y_test, y_val):
    thresh, _ = best_threshold_f1(val_probs, y_val)
    preds = (test_probs >= thresh).astype(int)
    return {
        "name":       name,
        "thresh":     thresh,
        "F1":         f1_score(y_test, preds, zero_division=0),
        "Precision":  precision_score(y_test, preds, zero_division=0),
        "Recall":     recall_score(y_test, preds, zero_division=0),
        "AUC-PR":     average_precision_score(y_test, test_probs),
        "preds":      preds,
        "test_probs": test_probs,
        "val_probs":  val_probs,
    }


# Base models individually
res_ffn  = evaluate("FFN",              ffn_test_probs,  ffn_val_probs,  y_test, y_val)
res_xgb  = evaluate("XGBoost",          xgb_test_probs,  xgb_val_probs,  y_test, y_val)
res_ae   = evaluate("Autoencoder",      ae_test_probs,   ae_val_probs,   y_test, y_val)
res_if   = evaluate("Isolation Forest", if_test_probs,   if_val_probs,   y_test, y_val)

# Meta-learners
meta_eval = {
    name: evaluate(f"Meta-{name}", d["test_probs"], d["val_probs"], y_test, y_val)
    for name, d in meta_results.items()
}

all_results = [res_ffn, res_xgb, res_ae, res_if] + list(meta_eval.values())

print(f"{'Model':<24}  {'Thresh':>7}  {'F1':>7}  {'Prec':>7}  {'Rec':>7}  {'AUC-PR':>8}")
print("-" * 72)
for r in all_results:
    tag = "  ◄ base" if r["name"] in {"FFN","XGBoost","Autoencoder","Isolation Forest"} else ""
    print(f"{r['name']:<24}  {r['thresh']:>7.3f}  {r['F1']:>7.4f}  {r['Precision']:>7.4f}  {r['Recall']:>7.4f}  {r['AUC-PR']:>8.4f}{tag}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for res, ls, lw in [(res_ffn, "--", 1.5), (res_xgb, "--", 1.5), (res_ae, "--", 1.5), (res_if, "--", 1.5)]:
    p, r, _ = precision_recall_curve(y_test, res["test_probs"])
    ax.plot(r, p, linestyle=ls, linewidth=lw, label=f"{res['name']} (AUC-PR={res['AUC-PR']:.3f})")

colors = plt.cm.tab10(np.linspace(0, 0.6, len(meta_eval)))
for (name, res), color in zip(meta_eval.items(), colors):
    p, r, _ = precision_recall_curve(y_test, res["test_probs"])
    ax.plot(r, p, color=color, linewidth=1.8, label=f"{res['name']} (AUC-PR={res['AUC-PR']:.3f})")

ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Test Set\n(dashed = base models, solid = meta-learners)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, res) in zip(axes.flat, meta_eval.items()):
    cm = confusion_matrix(y_test, res["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
    ax.set_title(f"{res['name']}\nF1={res['F1']:.4f}  AUC-PR={res['AUC-PR']:.4f}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices — Meta-Learners (Test Set)", y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

## 4. Cost-Sensitive Threshold Tuning

Minimize `10 × FN + 1 × FP` — missing a fraud costs 10× more than a false alarm.

In [ ]:
FN_COST, FP_COST = 10, 1

def cost_sensitive_threshold(val_probs, test_probs, y_val, y_test):
    thresholds = np.linspace(0.01, 0.99, 300)
    best_t, best_cost = 0.5, float("inf")
    for t in thresholds:
        preds = (val_probs >= t).astype(int)
        cm = confusion_matrix(y_val, preds, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        c = FN_COST * fn + FP_COST * fp
        if c < best_cost:
            best_cost, best_t = c, t
    preds_test = (test_probs >= best_t).astype(int)
    cm_test = confusion_matrix(y_test, preds_test, labels=[0, 1])
    tn, fp, fn, tp = cm_test.ravel()
    return best_t, FN_COST * fn + FP_COST * fp, f1_score(y_test, preds_test, zero_division=0)

print(f"{'Model':<24}  {'Threshold':>10}  {'Cost':>8}  {'F1':>7}")
print("-" * 56)
for r in all_results:
    t, cost, f1 = cost_sensitive_threshold(r["val_probs"], r["test_probs"], y_val, y_test)
    print(f"{r['name']:<24}  {t:>10.3f}  {cost:>8.0f}  {f1:>7.4f}")

## 5. Summary

In [ ]:
summary_df = pd.DataFrame([
    {"Model": r["name"], "F1": round(r["F1"],4), "Precision": round(r["Precision"],4),
     "Recall": round(r["Recall"],4), "AUC-PR": round(r["AUC-PR"],4)}
    for r in all_results
]).set_index("Model")

styled = summary_df.style.highlight_max(axis=0, props="font-weight: bold; color: darkgreen;").format("{:.4f}")
print("=== Test Set Results ===")
display(styled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ["F1", "Precision", "Recall", "AUC-PR"]
x = np.arange(len(metrics))
n, width = len(all_results), 0.8 / len(all_results)
colors = ["steelblue","darkorange","forestgreen","orchid"] + list(plt.cm.tab10(np.linspace(0, 0.6, len(meta_eval))))
for i, (r, color) in enumerate(zip(all_results, colors)):
    vals = [r["F1"], r["Precision"], r["Recall"], r["AUC-PR"]]
    axes[0].bar(x + i*width - (n-1)*width/2, vals, width, label=r["name"], color=color)
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.1); axes[0].set_ylabel("Score")
axes[0].set_title("All Models — Metric Comparison")
axes[0].legend(fontsize=7, ncol=2)

sns.heatmap(summary_df, annot=True, fmt=".4f", cmap="YlGn", linewidths=0.5, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Metric Heatmap")
plt.tight_layout(); plt.show()

best_meta = max(meta_eval.values(), key=lambda r: r["F1"])
print(f"\nBest meta-learner: {best_meta['name']}  F1={best_meta['F1']:.4f}  AUC-PR={best_meta['AUC-PR']:.4f}")

## 6. BankSim Generalizability Experiment

The full pipeline is re-run on BankSim to verify ensemble gains hold on a second dataset with different feature types.

**Download:** Kaggle → ealaxi/banksim1 → place `bs140513_032310.csv` in the project folder.

In [ ]:
# Class definitions needed by run_pipeline()
class FraudDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class FFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.BatchNorm1d(64),  nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.ReLU(), nn.Linear(32, 1),
        )
    def forward(self, x): return self.net(x).squeeze(1)

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, input_dim))
    def forward(self, x): return self.decoder(self.encoder(x))

def sigmoid(x): return 1 / (1 + np.exp(-x))

def get_recon_errors(model, X, batch_size=512):
    model.eval(); errors = []; X_t = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            xb = X_t[i:i+batch_size].to(DEVICE)
            errors.append(((xb - model(xb))**2).mean(dim=1).cpu().numpy())
    return np.concatenate(errors)

def get_ffn_probs(model, loader):
    model.eval(); probs = []
    with torch.no_grad():
        for xb, _ in loader:
            probs.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
    return np.concatenate(probs)

print("Helper classes defined.")

### 6.1 Load & Preprocess BankSim

In [ ]:
bs_df = pd.read_csv("bs140513_032310.csv")
for col in bs_df.select_dtypes("object").columns:
    bs_df[col] = bs_df[col].str.strip("'")

bs_counts = bs_df["fraud"].value_counts()
print(f"BankSim shape: {bs_df.shape}")
print(f"Normal: {bs_counts[0]:,}  Fraud: {bs_counts[1]:,}  ({bs_counts[1]/len(bs_df)*100:.2f}%)")

bs_df.drop(columns=[c for c in ["customer","merchant","zipcodeOri","zipMerchant"] if c in bs_df.columns], inplace=True)
bs_df = pd.get_dummies(bs_df, columns=["gender","category"], drop_first=False)
bs_scaler = StandardScaler()
bs_df[["step","age","amount"]] = bs_scaler.fit_transform(bs_df[["step","age","amount"]])

X_bs = bs_df[[c for c in bs_df.columns if c != "fraud"]].values.astype(np.float32)
y_bs = bs_df["fraud"].values.astype(np.float32)
INPUT_DIM_BS = X_bs.shape[1]

X_bs_train, X_bs_temp, y_bs_train, y_bs_temp = train_test_split(X_bs, y_bs, test_size=0.30, random_state=SEED, stratify=y_bs)
X_bs_val,   X_bs_test, y_bs_val,   y_bs_test = train_test_split(X_bs_temp, y_bs_temp, test_size=0.50, random_state=SEED, stratify=y_bs_temp)

print(f"\nTrain: {X_bs_train.shape[0]:,}  ({int(y_bs_train.sum())} fraud)")
print(f"Val:   {X_bs_val.shape[0]:,}   ({int(y_bs_val.sum())} fraud)")
print(f"Test:  {X_bs_test.shape[0]:,}   ({int(y_bs_test.sum())} fraud)")
print(f"Feature dim: {INPUT_DIM_BS}")

### 6.2 Run Pipeline on BankSim

In [ ]:
def run_pipeline(X_tr, y_tr, X_vl, y_vl, X_te, y_te, tag=""):
    import copy
    input_dim = X_tr.shape[1]
    neg_n, pos_n = (y_tr == 0).sum(), (y_tr == 1).sum()
    pw = torch.tensor([neg_n / pos_n], dtype=torch.float32).to(DEVICE)
    skf_p = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    # FFN OOF
    ffn_oof = np.zeros(len(X_tr), dtype=np.float32)
    for _, (tr_idx, vl_idx) in enumerate(skf_p.split(X_tr, y_tr)):
        X_f, y_f = X_tr[tr_idx], y_tr[tr_idx]
        fl = DataLoader(FraudDataset(X_f, y_f), batch_size=256, shuffle=True, num_workers=0)
        neg_f, pos_f = (y_f == 0).sum(), (y_f == 1).sum()
        fm = FFN(input_dim).to(DEVICE)
        fo = torch.optim.Adam(fm.parameters(), lr=1e-3, weight_decay=1e-5)
        fc = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg_f/pos_f], dtype=torch.float32).to(DEVICE))
        for _ in range(10):
            fm.train()
            for xb, yb in fl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                fo.zero_grad(); fc(fm(xb), yb).backward(); fo.step()
        fm.eval()
        with torch.no_grad():
            ffn_oof[vl_idx] = torch.sigmoid(fm(torch.tensor(X_tr[vl_idx]).to(DEVICE))).cpu().numpy()

    tr_loader = DataLoader(FraudDataset(X_tr, y_tr), batch_size=256, shuffle=True,  num_workers=0)
    vl_loader = DataLoader(FraudDataset(X_vl, y_vl), batch_size=512, shuffle=False, num_workers=0)
    te_loader = DataLoader(FraudDataset(X_te, y_te), batch_size=512, shuffle=False, num_workers=0)
    ffn_m = FFN(input_dim).to(DEVICE)
    ffn_o = torch.optim.Adam(ffn_m.parameters(), lr=1e-3, weight_decay=1e-5)
    ffn_c = nn.BCEWithLogitsLoss(pos_weight=pw)
    best_vl, best_sd = float("inf"), None
    for _ in range(20):
        ffn_m.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            ffn_o.zero_grad(); ffn_c(ffn_m(xb), yb).backward(); ffn_o.step()
        ffn_m.eval()
        vl_loss = sum(ffn_c(ffn_m(xb.to(DEVICE)), yb.to(DEVICE)).item()*len(xb) for xb, yb in vl_loader) / len(X_vl)
        if vl_loss < best_vl:
            best_vl = vl_loss; best_sd = {k: v.clone() for k, v in ffn_m.state_dict().items()}
    ffn_m.load_state_dict(best_sd); ffn_m.eval()
    ffn_vl, ffn_te = get_ffn_probs(ffn_m, vl_loader), get_ffn_probs(ffn_m, te_loader)

    # XGBoost OOF
    xp = dict(n_estimators=300, max_depth=6, learning_rate=0.05, scale_pos_weight=neg_n/pos_n,
              eval_metric="aucpr", random_state=SEED, n_jobs=-1, verbosity=0)
    xgb_oof = np.zeros(len(X_tr), dtype=np.float32)
    for _, (tr_idx, vl_idx) in enumerate(skf_p.split(X_tr, y_tr)):
        clf = xgb.XGBClassifier(**xp); clf.fit(X_tr[tr_idx], y_tr[tr_idx])
        xgb_oof[vl_idx] = clf.predict_proba(X_tr[vl_idx])[:, 1]
    xgb_m = xgb.XGBClassifier(**xp); xgb_m.fit(X_tr, y_tr)
    xgb_vl, xgb_te = xgb_m.predict_proba(X_vl)[:, 1], xgb_m.predict_proba(X_te)[:, 1]

    # Autoencoder
    X_normal = X_tr[y_tr == 0]
    ae_m = Autoencoder(input_dim).to(DEVICE)
    ae_o = torch.optim.Adam(ae_m.parameters(), lr=1e-3); ae_c = nn.MSELoss()
    ae_loader = DataLoader(FraudDataset(X_normal, np.zeros(len(X_normal), dtype=np.float32)),
                           batch_size=256, shuffle=True, num_workers=0)
    for _ in range(20):
        ae_m.train()
        for xb, _ in ae_loader:
            xb = xb.to(DEVICE); ae_o.zero_grad(); ae_c(ae_m(xb), xb).backward(); ae_o.step()
    aes = StandardScaler()
    ae_tr_p = sigmoid(aes.fit_transform(get_recon_errors(ae_m, X_tr).reshape(-1,1)).ravel())
    ae_vl_p = sigmoid(aes.transform(get_recon_errors(ae_m, X_vl).reshape(-1,1)).ravel())
    ae_te_p = sigmoid(aes.transform(get_recon_errors(ae_m, X_te).reshape(-1,1)).ravel())

    # Isolation Forest
    iso_m = IsolationForest(n_estimators=200, contamination="auto", random_state=SEED, n_jobs=-1)
    iso_m.fit(X_normal)
    ifs = StandardScaler()
    if_tr_p = sigmoid(ifs.fit_transform((-iso_m.score_samples(X_tr)).reshape(-1,1)).ravel())
    if_vl_p = sigmoid(ifs.transform((-iso_m.score_samples(X_vl)).reshape(-1,1)).ravel())
    if_te_p = sigmoid(ifs.transform((-iso_m.score_samples(X_te)).reshape(-1,1)).ravel())

    # Stack & train meta-learners
    m_tr = np.column_stack([ffn_oof, xgb_oof, ae_tr_p, if_tr_p])
    m_vl = np.column_stack([ffn_vl,  xgb_vl,  ae_vl_p, if_vl_p])
    m_te = np.column_stack([ffn_te,  xgb_te,  ae_te_p, if_te_p])

    results = [
        evaluate(f"{tag}FFN",              ffn_te,  ffn_vl,  y_te, y_vl),
        evaluate(f"{tag}XGBoost",          xgb_te,  xgb_vl,  y_te, y_vl),
        evaluate(f"{tag}Autoencoder",      ae_te_p, ae_vl_p, y_te, y_vl),
        evaluate(f"{tag}Isolation Forest", if_te_p, if_vl_p, y_te, y_vl),
    ]
    for name, clf_def in meta_learner_defs.items():
        clf_copy = copy.deepcopy(clf_def); clf_copy.fit(m_tr, y_tr)
        results.append(evaluate(f"{tag}Meta-{name}",
                                clf_copy.predict_proba(m_te)[:, 1],
                                clf_copy.predict_proba(m_vl)[:, 1], y_te, y_vl))
    return results

print("run_pipeline() defined.")

In [ ]:
print("Running full pipeline on BankSim (this will take a few minutes)...\n")
bs_results = run_pipeline(X_bs_train, y_bs_train, X_bs_val, y_bs_val, X_bs_test, y_bs_test, tag="BS-")

print(f"\n{'Model':<32}  {'F1':>7}  {'Prec':>7}  {'Rec':>7}  {'AUC-PR':>8}")
print("-" * 66)
for r in bs_results:
    print(f"{r['name']:<32}  {r['F1']:>7.4f}  {r['Precision']:>7.4f}  {r['Recall']:>7.4f}  {r['AUC-PR']:>8.4f}")

In [ ]:
# Cross-dataset comparison: CreditCard vs BankSim side-by-side
cc_map = {r["name"]: r for r in all_results}
bs_map = {r["name"].replace("BS-", ""): r for r in bs_results}
shared_names = list(bs_map.keys())

cc_f1 = [cc_map[n]["F1"]    for n in shared_names]
bs_f1 = [bs_map[n]["F1"]    for n in shared_names]
cc_ap = [cc_map[n]["AUC-PR"] for n in shared_names]
bs_ap = [bs_map[n]["AUC-PR"] for n in shared_names]

x = np.arange(len(shared_names))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(x - w/2, cc_f1, w, label="CreditCard", color="steelblue")
axes[0].bar(x + w/2, bs_f1, w, label="BankSim",    color="darkorange")
axes[0].set_xticks(x); axes[0].set_xticklabels(shared_names, rotation=30, ha="right")
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("F1 Score")
axes[0].set_title("F1 Score — CreditCard vs BankSim")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
axes[0].axvline(x=3.5, color="gray", linestyle="--", linewidth=0.8)
axes[0].text(1.5, 1.02, "Base Models", ha="center", fontsize=8, color="gray")
axes[0].text(7.5, 1.02, "Meta-Learners", ha="center", fontsize=8, color="gray")

axes[1].bar(x - w/2, cc_ap, w, label="CreditCard", color="steelblue")
axes[1].bar(x + w/2, bs_ap, w, label="BankSim",    color="darkorange")
axes[1].set_xticks(x); axes[1].set_xticklabels(shared_names, rotation=30, ha="right")
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("AUC-PR")
axes[1].set_title("AUC-PR — CreditCard vs BankSim")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)
axes[1].axvline(x=3.5, color="gray", linestyle="--", linewidth=0.8)

plt.suptitle("Generalizability: Ensemble Gains Across Datasets", fontsize=13)
plt.tight_layout(); plt.show()

# Summary table
rows = []
for n in shared_names:
    rows.append({
        "Model":       n,
        "CC F1":       round(cc_map[n]["F1"],    4),
        "CC AUC-PR":   round(cc_map[n]["AUC-PR"],4),
        "BS F1":       round(bs_map[n]["F1"],    4),
        "BS AUC-PR":   round(bs_map[n]["AUC-PR"],4),
        "ΔF1 (BS-CC)": round(bs_map[n]["F1"]     - cc_map[n]["F1"],    4),
    })

comp_df = pd.DataFrame(rows).set_index("Model")
styled_comp = comp_df.style.background_gradient(cmap="RdYlGn", subset=["CC F1","BS F1","CC AUC-PR","BS AUC-PR"], vmin=0, vmax=1)
print("=== Cross-Dataset Comparison ===")
display(styled_comp)